# Part 1i: Hyperparameter Tuning with Keras Tuner
**Author:** Kalhar Mayurbhai Patel (019140511)

Keras Tuner automates hyperparameter search. We demonstrate RandomSearch and Hyperband.

In [1]:
!pip install keras-tuner -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 852.1 kB/s eta 0:00:00


In [2]:
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

data = fetch_california_housing()
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## Define the Hypermodel

In [3]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import keras_tuner as kt

def build_hypermodel(hp):
    model = keras.Sequential()
    # Tune number of layers
    for i in range(hp.Int('n_layers', 1, 4)):
        model.add(layers.Dense(
            units=hp.Choice(f'units_{i}', [32, 64, 128, 256]),
            activation=hp.Choice('activation', ['relu', 'selu', 'elu'])
        ))
        if hp.Boolean('use_dropout'):
            model.add(layers.Dropout(hp.Float('dropout', 0.1, 0.5, step=0.1)))
        if hp.Boolean('use_batchnorm'):
            model.add(layers.BatchNormalization())

    model.add(layers.Dense(1))  # regression output
    lr = hp.Float('learning_rate', 1e-4, 1e-2, sampling='log')
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),
                  loss='mse', metrics=['mae'])
    return model

## Run RandomSearch

In [4]:
import shutil, os
if os.path.exists('kt_random'): shutil.rmtree('kt_random')

tuner_random = kt.RandomSearch(
    build_hypermodel,
    objective='val_mae',
    max_trials=10,
    directory='kt_random',
    project_name='housing'
)

tuner_random.search(X_train, y_train, epochs=30, validation_data=(X_test, y_test),
                    verbose=0, callbacks=[keras.callbacks.EarlyStopping('val_loss', patience=5)])

best_hp = tuner_random.get_best_hyperparameters()[0]
print("\nBest Hyperparameters (RandomSearch):")
for key in ['n_layers', 'activation', 'use_dropout', 'use_batchnorm', 'learning_rate']:
    try:
        print(f"  {key}: {best_hp.get(key)}")
    except: pass

best_model = tuner_random.get_best_models()[0]
_, mae = best_model.evaluate(X_test, y_test, verbose=0)
print(f"Best Model Val MAE: {mae:.4f}")


Best Hyperparameters (RandomSearch):
  n_layers: 3
  activation: relu
  use_dropout: False
  use_batchnorm: False
  learning_rate: 0.004490277088961167


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Best Model Val MAE: 0.3430


## Run Hyperband (faster, early-stop based)

In [7]:
if os.path.exists('kt_hyperband'): shutil.rmtree('kt_hyperband')

tuner_hb = kt.Hyperband(
    build_hypermodel,
    objective='val_mae',
    max_epochs=3,
    factor=3,
    directory='kt_hyperband',
    project_name='housing'
)

tuner_hb.search(X_train, y_train, epochs=3, validation_data=(X_test, y_test), verbose=0)

best_hp_hb = tuner_hb.get_best_hyperparameters()[0]
print("\nBest Hyperparameters (Hyperband):")
for key in ['n_layers', 'activation', 'use_dropout', 'learning_rate']:
    try:
        print(f"  {key}: {best_hp_hb.get(key)}")
    except: pass

best_hb = tuner_hb.get_best_models()[0]
_, mae_hb = best_hb.evaluate(X_test, y_test, verbose=0)
print(f"Best Hyperband Model MAE: {mae_hb:.4f}")


Best Hyperparameters (Hyperband):
  n_layers: 4
  activation: selu
  use_dropout: False
  learning_rate: 0.00104935888624918
Best Hyperband Model MAE: 0.4132


## Summary
- **RandomSearch**: samples random combinations, simple but can miss good combos
- **Hyperband**: allocates budget efficiently by early-stopping bad trials
- **BayesianOptimization** (also available in kt): builds a surrogate model to guide search
- Define search spaces with `hp.Int`, `hp.Float`, `hp.Choice`, `hp.Boolean`